In [ ]:
import os

import polars as pl

from social_groups.reporting.analysis_columns import AnalysisColumn
from social_groups.reporting.group_reply import GroupReplyAggregator, MajorityVote
from social_groups.reporting.parsing import (
    AnswerOptions,
    AnswerParser,
    AnswerComparer,
)
from social_groups.directories import REPORTING_DIR
from social_groups.reporting.plots.config import MODEL_NAME_TO_LETTER_MAPPING
from social_groups.reporting.group_decision_scheme import calculate_extended_decision_scheme, \
    calculate_decision_scheme
from social_groups.reporting.heterogenous_comparison.data_retrieval import get_baseline_frame, get_mad_frame, \
    apply_parsing_and_group_decision

%load_ext autoreload
%autoreload 2

In [ ]:
output_dir = REPORTING_DIR / "heterogeneous_group"
os.makedirs(output_dir, exist_ok=True)

triple_underscore_handling = "random"

In [ ]:
parser = AnswerParser(AnswerOptions.letters_A_to_J)
group_reply = GroupReplyAggregator(MajorityVote())
comparer = AnswerComparer(
    AnswerOptions.letters_A_to_J, triple_underscore_handling="random"
)

In [ ]:
baseline_frame = get_baseline_frame()

In [ ]:
baseline_frame

In [ ]:
print("Number of unparsable answers:")
print(
    baseline_frame.with_columns(
        parser(pl.col("final_answer")).alias(AnalysisColumn.parsed_answer.value)
    )
    .group_by("model_name")
    .agg(
        no_null=pl.col(AnalysisColumn.parsed_answer.value)
        .str.starts_with("___")
        .not_()
        .sum(),
        null_percentage=(
                pl.col(AnalysisColumn.parsed_answer.value).str.starts_with("___").mean()
                * 100
        ).round(2),
    )
    .sort(pl.col("model_name").str.extract(r"-(\d+\.?\d*)B", 1).cast(pl.Float64))
)

baseline_analysis = (
    baseline_frame.with_columns(
        parser(pl.col("final_answer")).alias(AnalysisColumn.parsed_answer.value)
    )
    .with_columns(
        is_correct=comparer(
            pl.col(AnalysisColumn.parsed_answer.value),
            pl.col("answer_string")
        )
    )
    .group_by("model_name")
    .agg(accuracy=pl.col("is_correct").mean())
    .sort(pl.col("model_name").str.extract(r"-(\d+\.?\d*)B", 1).cast(pl.Float64))
)

baseline_analysis

## Analzying Basic Group Behaviour

In [ ]:
mad_frame = get_mad_frame()
mad_frame

In [ ]:
model_name_sort = {
    "Qwen/Qwen3-14B": 1,
    "Qwen/Qwen3-4B": 2,
    "Qwen/Qwen3-0.6B": 3,
}

mad_analysis = apply_parsing_and_group_decision(mad_frame, parser, comparer, group_reply)

table_page_46 = pl.concat(
    [
        (
            mad_analysis.group_by("group_constellation")
            .agg(accuracy=pl.col("is_correct").mean())
            .with_columns(origin=pl.lit("(MAD)"))
        ),
        (
            baseline_analysis.with_columns(
                group_constellation=pl.col("model_name").replace(MODEL_NAME_TO_LETTER_MAPPING),
                origin=pl.lit("(baseline)"),
            ).drop("model_name")
        ),
    ],
    how="diagonal",
)

table_page_46.write_csv(output_dir / "llm_based_table_page_46.csv")

table_page_46

In [ ]:
original_table_page46 = pl.DataFrame(
    {
        "group_constellation": [
            "HHH",
            "HHM",
            "HHL",
            "HML",
            "HMM",
            "H",
            "HLL",
            "MMM",
            "M",
            "MML",
            "MLL",
            "L",
            "LLL",
        ],
        "score (-115 to 115)": [80, 74, 67, 64, 61, 60, 56, 48, 42, 39, 37, 25, 21],
    }
)
original_table_page46.write_csv(output_dir / "human_based_table_page_46.csv")
original_table_page46

In [ ]:
decision_schemes = mad_analysis.group_by("group_constellation").map_groups(
    lambda g: calculate_decision_scheme(
        g,
        AnalysisColumn.parsed_individual_answers_before.value,
        AnalysisColumn.parsed_individual_answers_after.value,
        "answer_string",
        group_reply,
        comparer,
    ).select(
        pl.lit(g["group_constellation"].unique().item()).alias("group_constellation"),
        "Correct Members Beginning",
        "correct",
        "incorrect",
    )
).sort("group_constellation")

decision_schemes.write_csv(output_dir / "group_decision_schemes.csv")

decision_schemes

In [ ]:
from social_groups.reporting.plots.decision_scheme_extended import make_decision_scheme_extended_plot

for group in mad_analysis["group_constellation"].unique().sort():
    extended_ds = calculate_extended_decision_scheme(
        mad_analysis.filter(pl.col("group_constellation") == group),
        AnalysisColumn.parsed_individual_answers_before.value,
        AnalysisColumn.parsed_individual_answers_after.value,
        "answer_string",
        group_reply,
        comparer,
    )
    make_decision_scheme_extended_plot(extended_ds, title=f"Group {group}").show()

In [ ]:
calculate_decision_scheme(mad_analysis.filter(pl.col("group_constellation") == "HH"),
                          AnalysisColumn.parsed_individual_answers_before.value,
                          AnalysisColumn.parsed_individual_answers_after.value,
                          "answer_string",
                          group_reply,
                          comparer, )

In [30]:
mad_analysis.select("phoenix_span_id", "run_identifier")

phoenix_span_id,run_identifier
str,str
"""65d141b20d77b78d""","""2026-02-02-11-11-12 - heteroge…"
"""8b9615c1d204a1a8""","""2026-02-02-11-11-12 - heteroge…"
"""e810274adba8434c""","""2026-02-02-11-11-12 - heteroge…"
"""5a0a84303632c13f""","""2026-02-02-11-11-12 - heteroge…"
"""7fb4a5cac1ce54e2""","""2026-02-02-11-11-12 - heteroge…"
…,…
"""4f1f24d37b56c4f7""","""2026-02-02-11-19-10 - heteroge…"
"""090f156666427bd9""","""2026-02-02-11-19-10 - heteroge…"
"""e344fb047c18e259""","""2026-02-02-11-19-10 - heteroge…"


## Analysis: Is the group finding answers "together" ? (e.g. can it find answers out of wrong start)